# Chương 8 – DPO và Alignment (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/phuongtnd1994/ptnd-ModelAdaptationBook/blob/main/code/chapter08/chapter08_colab.ipynb)

Full-parameter DPO cần ~54 GB VRAM (không vừa Colab thông thường), nên notebook này:

1. Train **LoRA-DPO** (`--lora`, ~11 GB) trên nền model SFT chương 6 — vừa L4.
2. Đánh giá/inference model **full DPO** đã publish trên Hugging Face.
3. Dùng **Claude làm LLM-judge** để so sánh base / SFT / DPO / LoRA-DPO (thay cho OpenRouter trong sách).

## 0. Cài đặt môi trường

Chạy lần lượt các ô dưới đây **một lần** mỗi khi mở runtime mới. Ô đầu tiên clone repo, cài package của sách và tạo hàm `run()` để chạy lệnh và in log trực tiếp ra notebook.

In [ ]:
import os, sys, subprocess

# Repo chứa code của sách. Nếu fork của bạn để private, đổi sang repo gốc:
#   https://github.com/bahree/ModelAdaptationBook.git
REPO_URL = "https://github.com/phuongtnd1994/ptnd-ModelAdaptationBook.git"
REPO_DIR = "/content/ModelAdaptationBook"
CODE_DIR = f"{REPO_DIR}/code"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(CODE_DIR)
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

os.environ["PYTHONUNBUFFERED"] = "1"   # log của script con hiện ra ngay
os.environ["WANDB_DISABLED"] = "true"  # không dùng Weights & Biases trên Colab


def run(cmd: str) -> None:
    """Chạy một lệnh shell từ thư mục code/, in log trực tiếp, báo lỗi nếu thất bại."""
    print(f"$ {cmd}\n", flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=CODE_DIR, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"Lệnh thất bại (exit code {p.returncode}): {cmd}")


run('pip install -q -e ".[dev]"')
print("\nĐã cài xong. Thư mục làm việc:", os.getcwd())

In [ ]:
import torch

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    cap = torch.cuda.get_device_capability(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {name} | {mem:.1f} GB | compute capability {cap[0]}.{cap[1]}")
    if cap < (8, 0):
        print("CẢNH BÁO: GPU này (ví dụ T4) không hỗ trợ bf16 gốc. Các bước TRAINING "
              "trong sách dùng bf16 -> hãy đổi sang L4 hoặc A100 "
              "(Runtime > Change runtime type).")
else:
    print("Không có GPU. Chỉ các bước chạy trên CPU / gọi API hoạt động được.")

### Khoá API (Colab Secrets)

Mở biểu tượng 🔑 (Secrets) ở thanh bên trái, thêm:

- `ANTHROPIC_API_KEY` = API key Claude của bạn (**bắt buộc** cho notebook này)
- `HF_TOKEN` = token Hugging Face (không bắt buộc)

và bật *Notebook access*. Nếu chưa thêm, ô dưới sẽ hỏi bạn nhập tay.

In [ ]:
import os
from getpass import getpass


def get_secret(name, required=False):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    if os.environ.get(name):
        return os.environ[name]
    if required:
        return getpass(f"Nhập {name}: ")
    return None


os.environ["ANTHROPIC_API_KEY"] = get_secret("ANTHROPIC_API_KEY", required=True)
hf = get_secret("HF_TOKEN")  # không bắt buộc, giúp tải model nhanh hơn
if hf:
    os.environ["HF_TOKEN"] = hf
print("Đã nạp khoá:", [k for k in ("ANTHROPIC_API_KEY", "HF_TOKEN") if os.environ.get(k)])

### Tải model đã train sẵn từ Hugging Face

Tác giả sách đã publish model của từng chương tại [`bahree/ModelAdaptationBook`](https://huggingface.co/bahree/ModelAdaptationBook). Ô dưới tải đúng các thư mục cần cho chương này về `/content/hub/` để các script dùng như đường dẫn local (không cần tự train các chương trước).

In [ ]:
from huggingface_hub import snapshot_download

HUB_DIR = "/content/hub"
SUBFOLDERS = ["ch6-sft", "ch8-dpo"]
snapshot_download("bahree/ModelAdaptationBook", local_dir=HUB_DIR,
                  allow_patterns=[f"{s}/*" for s in SUBFOLDERS])
for s in SUBFOLDERS:
    print(s, "->", f"{HUB_DIR}/{s}", sorted(os.listdir(f"{HUB_DIR}/{s}"))[:6])

In [ ]:
SFT_DIR = "/content/hub/ch6-sft"
DPO_DIR = "/content/hub/ch8-dpo"

## 1. Unit test

In [ ]:
run("pytest chapter08/tests/ -q")

## 2. Chuẩn bị preference pairs (Listing 8.1)

In [ ]:
run("python -m chapter08.scripts.prepare_preference_data "
    "--preferences data/it_support/preferences.jsonl "
    "--eval_valid data/it_support/valid.jsonl "
    "--out chapter08/data/preference_pairs --num_valid 30")

## 3. Train LoRA-DPO (Listing 8.2, biến thể `--lora`)

~3–5 phút, ~11 GB VRAM.

In [ ]:
run(f"python -m chapter08.train_dpo --lora --model_dir {SFT_DIR} "
    "--train chapter08/data/preference_pairs/train.jsonl "
    "--valid chapter08/data/preference_pairs/valid.jsonl "
    "--out chapter08/runs/dpo_lora_run1")

## 4. Đánh giá SFT vs DPO (Listing 8.3) – dùng model full DPO đã publish

In [ ]:
run(f"python -m chapter08.eval_dpo --data_dir data/it_support --split test "
    f"--sft_dir {SFT_DIR} --dpo_dir {DPO_DIR} --output chapter08/eval/dpo_report_colab.json")

In [ ]:
import json, pathlib
p = pathlib.Path("chapter08/eval/dpo_report_colab.json")
print(json.dumps(json.loads(p.read_text()), indent=2, ensure_ascii=False)[:3000] if p.exists() else f"Chưa có {p}")

## 5. Safety regression sau DPO (Listing 8.4)

In [ ]:
run(f"python -m chapter08.scripts.safety_after_dpo --sft_dir {SFT_DIR} --dpo_dir {DPO_DIR} --output_dir chapter08/eval/safety_colab")

## 6. Inference

Model full DPO (Hub) và LoRA-DPO bạn vừa train (adapter gắn lên model SFT).

In [ ]:
Q = "How do I troubleshoot a VPN connection failure?"
run(f'python -m chapter08.generate --model_dir {DPO_DIR} --prompt "{Q}"')
run(f'python -m chapter07.generate --base_model {SFT_DIR} --adapter_dir chapter08/runs/dpo_lora_run1 --prompt "{Q}"')

## 7. (Bonus) Claude làm LLM-judge

`eval_lora_dpo` sinh câu trả lời từ base, SFT, DPO, LoRA-DPO rồi nhờ judge chấm điểm và so cặp. Sách dùng OpenRouter; shim dưới đây chuyển sang Claude (`CLAUDE_MODEL`, mặc định `claude-sonnet-5-5`). Số lời gọi API: vài trăm, chi phí ước tính vài USD.

In [ ]:
%%writefile /content/run_with_claude.py
"""Chạy một module của sách nhưng thay OpenRouter bằng Claude API (ANTHROPIC_API_KEY).

Dùng: python /content/run_with_claude.py <module> [args...]
Các script trong sách gọi common.openrouter.chat(); file này thay hàm đó bằng một hàm
gọi Anthropic SDK rồi chạy module như `python -m <module>`.
"""
import os
import runpy
import sys

sys.path.insert(0, os.getcwd())

import anthropic
import common.openrouter as openrouter

# Model Claude dùng cho mọi lời gọi không chỉ định rõ model Anthropic
# (ví dụ judge "openai/gpt-5.5" trong chương 8). Đổi qua biến môi trường CLAUDE_MODEL.
CLAUDE_MODEL = os.environ.get("CLAUDE_MODEL", "claude-sonnet-5-5")
_client = anthropic.Anthropic()


def _resolve(model):
    # "anthropic/claude-sonnet-5.5" -> "claude-sonnet-5-5"; model của hãng khác -> CLAUDE_MODEL
    if model and model.startswith("anthropic/"):
        return model.split("/", 1)[1].replace(".", "-")
    return CLAUDE_MODEL


def chat(messages, model, *, max_tokens=1024, temperature=0.0, timeout=60, extra_body=None):
    # temperature bị bỏ qua: các model Claude hiện tại không nhận tham số sampling.
    system = "\n\n".join(m["content"] for m in messages if m["role"] == "system")
    convo = [m for m in messages if m["role"] != "system"]
    kwargs = {"system": system} if system else {}
    resp = _client.messages.create(
        model=_resolve(model),
        max_tokens=max(max_tokens, 4096),  # chừa chỗ cho adaptive thinking
        output_config={"effort": "low"},
        messages=convo,
        **kwargs,
    )
    text = "".join(b.text for b in resp.content if b.type == "text")
    if not text:
        raise openrouter.OpenRouterError(f"Claude trả về rỗng (stop_reason={resp.stop_reason})")
    u = resp.usage
    return {
        "content": text,
        "reasoning": None,
        "usage": {"prompt_tokens": u.input_tokens, "completion_tokens": u.output_tokens,
                  "total_tokens": u.input_tokens + u.output_tokens, "cost": None},
        "model": resp.model,
        "raw": resp.to_dict(),
    }


openrouter.chat = chat
os.environ.setdefault("OPENROUTER_API_KEY", "not-used-claude-shim")

module = sys.argv[1]
sys.argv = [module] + sys.argv[2:]
runpy.run_module(module, run_name="__main__", alter_sys=True)

In [ ]:
os.environ["CLAUDE_MODEL"] = "claude-sonnet-5-5"   # hoặc "claude-opus-5-5"
run(f"python /content/run_with_claude.py chapter08.eval_lora_dpo "
    f"--sft {SFT_DIR} --dpo {DPO_DIR} --lora_dpo chapter08/runs/dpo_lora_run1 "
    "--workers 4 --out chapter08/eval/dpo_lora_report_claude.json")

In [ ]:
import json, pathlib
p = pathlib.Path("chapter08/eval/dpo_lora_report_claude.json")
print(json.dumps(json.loads(p.read_text()), indent=2, ensure_ascii=False)[:4000] if p.exists() else f"Chưa có {p}")

## 8. (Chỉ khi có GPU 80 GB) Full-parameter DPO

Cần H100/A100 80 GB. Trên Colab thường không có — bỏ qua.

In [ ]:
vram = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
if vram >= 79:
    run(f"python -m chapter08.train_dpo --model_dir {SFT_DIR} "
        "--train chapter08/data/preference_pairs/train.jsonl "
        "--valid chapter08/data/preference_pairs/valid.jsonl --out chapter08/runs/dpo_run1")
else:
    print(f"Bỏ qua: GPU có {vram:.0f} GB, full DPO cần ~54 GB trở lên.")

## Lưu kết quả về Google Drive (tuỳ chọn)

Runtime Colab sẽ bị xoá khi ngắt kết nối. Chạy ô dưới để copy thư mục kết quả sang Drive.

In [ ]:
from google.colab import drive
import shutil

drive.mount("/content/drive")
DEST = "/content/drive/MyDrive/ModelAdaptationBook"
for p in ["chapter08/runs/dpo_lora_run1", "chapter08/eval"]:
    src = os.path.join(CODE_DIR, p)
    dst = os.path.join(DEST, p)
    if os.path.isdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print("Đã lưu", p, "->", dst)
    elif os.path.isfile(src):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(src, dst)
        print("Đã lưu", p, "->", dst)